In [ ]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [ ]:
from langchain_core.documents import Document
from langchain_community.document_loaders.text import TextLoader
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
sample_doc = Document(
    page_content ="Hello world!",
    metadata = {"source" : "https:www.google.com"}
)

print(type(sample_doc))

In [ ]:
loader = TextLoader("Python.txt",encoding="utf-8")

In [ ]:
document = loader.load()
document

# Load the pdf

In [ ]:
pdf_loader = PyPDFLoader("research2.pdf")

In [ ]:
pdf_document = pdf_loader.load()
pdf_document

# Ingestion Pipeline

In [ ]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [ ]:
def load_all_pdf():
  floder_path = "/content/Data"
  num_docs = 0
  all_docs = []

  for filename in os.listdir(floder_path):
    if filename.lower().endswith(".pdf"):
      pdf_path = os.path.join(floder_path,filename)

      loader = PyPDFLoader(pdf_path)
      doc = loader.load()

      all_docs.extend(doc)

      num_docs = num_docs + 1
  print("Total_pdf :",num_docs)
  print("Total_pages :",len(all_docs))

  return all_docs


In [ ]:
all_pdf_documents = load_all_pdf()

Create chunks on the pdf

In [ ]:
#!pip install langchain_text_splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
def create_chunks(documents,chunk_size = 500,chunk_overlap = 50):
  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size = chunk_size,
      chunk_overlap = chunk_overlap
  )

  chunked_docs = text_splitter.split_documents(documents)

  return chunked_docs


Chunks

In [ ]:
pdf_chunk = create_chunks(all_pdf_documents)

In [ ]:
len(pdf_chunk)

# Embeddings

In [ ]:
pip install -U sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer


In [ ]:
class EmbeddingManager:
  def __init__(self,model_name="all-MiniLM-L6-v2"):
    self.model_name = model_name
    print("Loading model....",self.model_name)

    self.model = SentenceTransformer(self.model_name)
    print("embeddings dimension :",self.model.get_sentence_embedding_dimension())

  def generate_embeddings(self,text):
    embeddings = self.model.encode(text,show_progress_bar = True)
    print("Embedding shape :",embeddings.shape)

    return embeddings

In [ ]:
embeddingmanager =  EmbeddingManager()

# Vector Store

In [ ]:
import chromadb
import uuid

In [ ]:
class VectorStoreManager:
  def __init__(self,persist_directory = "Data/vector_store",collection_name = "pdf_documents"):
    self.persist_directory = persist_directory
    self.collection_name = collection_name
    self.collection = None
    self.client = None
    self._initialize_store()

  def _initialize_store(self):
    os.makedirs(self.persist_directory,exist_ok=True)

    # select a client

    self.client = chromadb.PersistentClient(path=self.persist_directory)

    self.collection = self.client.get_or_create_collection(
        name = self.collection_name,
        metadata = {"description": "Vector store collection for pdf embeddings in RAG"}
    )

    print("initialized vector store with collection ", self.collection_name)
    print("Documents in collection :",self.collection.count())

  def add_documents(self,documents,embeddings):
    if len(documents) != len(embeddings):
      raise ValueError("number of documents doesn't match number of embeddings")

    # store =>  ids,embeddings,document,metadata

    ids = []
    all_metadata = []
    document_content = []
    embeddings_list = []

    for i, (doc, emb) in enumerate(zip(documents,embeddings)):
      doc_id = f"doc_{uuid.uuid4()}"
      ids.append(doc_id)

      metadata = dict(doc.metadata)
      metadata["doc_index"] = i
      metadata['content_length'] = len(doc.page_content)
      all_metadata.append(metadata)

      document_content.append(doc.page_content)
      embeddings_list.append(emb.tolist())

    self.collection.add(
        ids = ids,
        metadatas = all_metadata,
        documents = document_content,
        embeddings = embeddings_list
    )

In [ ]:
vector_store = VectorStoreManager()

# data -> Documents -> Chunks -> Embeddings -> Store in Vector Store

In [ ]:
# data -> Documents -> Chunks -> Embeddings -> Store in Vector Store

texts = [document.page_content for document in pdf_chunk]
embeddings = embeddingmanager.generate_embeddings(texts)
vector_store.add_document(pdf_chunk, embeddings)